In [ ]:
import torch, time, gc, warnings
from transformers import AutoModelForCausalLM, logging
import modelopt.torch.quantization as mtq

warnings.filterwarnings("ignore")
logging.set_verbosity_error()

model_id = "Qwen/Qwen3-4B-Thinking-2507"
inp = torch.randint(1, 10000, (1, 1000), device="cuda")
calib = torch.randint(1, 10000, (1, 8), device="cuda")
res = []

recipes = [
    ("FP16_Baseline", None),
    ("FP8", mtq.FP8_DEFAULT_CFG),
    ("INT8_Smooth", mtq.INT8_SMOOTHQUANT_CFG),
    ("INT4_AWQ", mtq.INT4_AWQ_CFG),
    ("AutoQuantize", [mtq.FP8_DEFAULT_CFG, mtq.INT4_AWQ_CFG]),
]

/home/student/miniconda3/envs/hpml_lab/lib/python3.10/site-packages/modelopt/torch/utils/logging.py:116: UserWarning: Failed to import modelopt transformers plugin due to: ImportError("Unable to import required dependencies:\npytz: No module named 'pytz'"). You may ignore this warning if you do not need this plugin.
  warnings.warn(message, *args, **kwargs)
/home/student/miniconda3/envs/hpml_lab/lib/python3.10/site-packages/modelopt/torch/utils/logging.py:116: UserWarning: Failed to import modelopt huggingface plugin due to: ImportError("cannot import name 'ModelOptHFTrainer' from 'modelopt.torch.opt.plugins' (/home/student/miniconda3/envs/hpml_lab/lib/python3.10/site-packages/modelopt/torch/opt/plugins/__init__.py)"). You may ignore this warning if you do not need this plugin.
  warnings.warn(message, *args, **kwargs)
/home/student/miniconda3/envs/hpml_lab/lib/python3.10/site-packages/modelopt/torch/utils/logging.py:116: UserWarning: Failed to import modelopt transformers trainer plug

In [2]:
for name, cfg in recipes:
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="cuda")

    if cfg:
        if name == "AutoQuantize":
            mtq.auto_quantize(
                model,
                constraints={"effective_bits": 6.0},
                quantization_formats=cfg,
                data_loader=[calib],
                forward_step=lambda m, b: m(b),
                forward_backward_step=lambda m, b: m(b, labels=b).loss.backward(),
            )
        else:
            mtq.quantize(model, config=cfg, forward_loop=lambda m: m(calib))

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    t0 = time.time()
    model.generate(inp, attention_mask=torch.ones_like(inp), max_new_tokens=3, min_new_tokens=3, pad_token_id=0)
    res.append(
        f"{name:15} | Time: {time.time() - t0:.3f} s | Peak Memory: {torch.cuda.max_memory_allocated() / 1024**2:.0f} MB"
    )

    del model
    gc.collect()

print("\n".join(res))

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Registered <class 'transformers.models.qwen3.modeling_qwen3.Qwen3Attention'> to _QuantAttention for KV Cache quantization
Inserted 903 quantizers
Loading extension modelopt_cuda_ext_fp8...
Loaded extension modelopt_cuda_ext_fp8 in 0.0 seconds


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Inserted 903 quantizers
Smoothed 252 modules
Loading extension modelopt_cuda_ext...
Loaded extension modelopt_cuda_ext in 0.0 seconds


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

... (6 outputs left out)

AutoQuantize: starting score estimation,  memory (MB) | allocated:  7.81e+03 | max_allocated:  7.81e+03 | reserved:  9.62e+03 | max_reserved:  9.62e+03


Estimating auto_quantize scores:   1%|          | 1/128 [00:00<00:37,  3.40it/s]

AutoQuantize: After score estimation memory (MB) | allocated:  7.86e+03 | max_allocated:  1.01e+04 | reserved:  1.04e+04 | max_reserved:  1.04e+04


FP16_Baseline   | Time: 0.465 s | Peak Memory: 7920 MB
FP8             | Time: 0.314 s | Peak Memory: 8463 MB
INT8_Smooth     | Time: 0.308 s | Peak Memory: 8480 MB
INT4_AWQ        | Time: 0.282 s | Peak Memory: 8571 MB
AutoQuantize    | Time: 0.285 s | Peak Memory: 8709 MB
